# Extractive text summarization — build it from scratch

**NLP · practice topic 02**

Open `explainer.html` first. This notebook walks the same path, but you write the code.

## The problem

Given a news article, return the **three sentences that best summarize it** — chosen by the computer, not by you.
Then prove, with a number, that your method beats doing nothing clever.

You will build three summarizers and measure all three:

| # | Method | The idea |
|---|--------|----------|
| 1 | **Lead-3** | Take the first three sentences. No maths. This is the baseline you must beat. |
| 2 | **TF-IDF** | Score each sentence by how many *informative* words it contains. |
| 3 | **TextRank** | Let sentences vote for each other. PageRank, on sentences instead of web pages. |

Then score all three with **ROUGE-1** against a human-written reference summary.

## Rules of the game

- Only `numpy`, `re` and the standard library. No downloads, no `pip install`, runs in under a second.
- Every `# TODO` is a line *you* write. The boilerplate around it is already done.
- Work top to bottom. Each cell ends with an `assert` check that tells you whether you got it right.
- Stuck on a TODO for more than a few minutes? Re-read the matching section of `explainer.html` — the
  section number is in each heading.

---
## Cell 0 — Setup and the article

Nothing to write. Run it and read the article; you will be looking at these eight sentences all the way down.

In [ ]:
import re
import numpy as np

np.set_printoptions(precision=3, suppress=True)

ARTICLE = (
    "The city council approved a new flood barrier on Tuesday after two years of delays. "
    "The barrier will run for three kilometres along the eastern riverbank. "
    "Engineers say the design can hold back water levels of up to four metres. "
    "Funding for the barrier was delayed twice, once in 2023 and again last spring. "
    "Residents of the eastern district have pushed for a flood barrier since 2021. "
    "The 2021 flood damaged more than nine hundred homes and closed two schools. "
    "Construction of the barrier is expected to begin in March and finish within two years. "
    "The council said the final cost would be confirmed once contracts are signed."
)

# A human-written summary of the same article. We score our output against this at the very end.
REFERENCE = (
    "The city council approved a new flood barrier after two years of delays. "
    "Residents of the eastern district have pushed for a flood barrier since the 2021 flood "
    "damaged nine hundred homes. Construction is expected to begin in March."
)

print(len(ARTICLE.split()), "words in the article")
print(len(REFERENCE.split()), "words in the reference summary")

---
## Cell 1 — Split into sentences  ·  *explainer section 3*

We need a **list of sentences** before we can pick any. The rule: cut wherever a `.`, `?` or `!` is followed by
whitespace and then a capital letter.

The pattern uses two *lookarounds* — bits that check what sits beside a position without consuming it:

- `(?<=[.!?])` — **lookbehind**: "there must be a `.`, `!` or `?` just before here"
- `(?=[A-Z])` — **lookahead**: "there must be a capital letter just after here"
- `\s+` in the middle — the whitespace that actually gets eaten by the split

Read as a whole: *split on the run of whitespace sitting between sentence-ending punctuation and a capital letter.*
Using lookarounds is what keeps the `.` attached to the sentence it ends.

In [ ]:
def split_sentences(text):
    """Split `text` into a list of sentences."""
    # TODO 1a: build the pattern described above and split `text` with it.
    #          Hint: re.split(r"...", text) -- one line.
    parts = None

    # Done for you: tidy the whitespace and drop anything empty.
    return [p.strip() for p in parts if p.strip()]


sentences = split_sentences(ARTICLE)
N = len(sentences)

for i, s in enumerate(sentences):
    print("s{}: {}".format(i, s))

assert N == 8, "expected 8 sentences, got {}".format(N)
assert sentences[0].endswith("delays."), "the full stop should stay attached to the sentence"
print("\ncheck passed --", N, "sentences")

---
## Cell 2 — Tokenize  ·  *explainer section 4*

Turn a sentence into a clean list of **tokens** — the words the computer will actually count.

Three moves, in this order:

1. **Lowercase**, so `Barrier` and `barrier` become one word.
2. **Replace every non-letter, non-digit with a space**, so `barrier,` becomes `barrier`. Use
   `re.sub(r"[^a-z0-9\s]", " ", text)` — and note this only works *after* lowercasing, because `A-Z` is not in
   the keep-set.
3. **Drop stopwords** — words so common they say nothing about which sentence is special.

In [ ]:
STOPWORDS = set("""
a an the and or but if then than that this these those of to in on at for with by from as is are was
were be been being it its will would can could may might have has had do does did not no so up out over under
about into more most other some such only own same too very s t just
""".split())


def tokenize(text):
    """Lowercase, strip punctuation, split on whitespace, drop stopwords."""
    # TODO 2a: lowercase `text`, then replace every character that is not a-z, 0-9
    #          or whitespace with a space.
    cleaned = None

    # TODO 2b: split `cleaned` on whitespace and keep only words NOT in STOPWORDS.
    #          A list comprehension is the natural fit.
    tokens = None

    return tokens


tokens = [tokenize(s) for s in sentences]

for i, t in enumerate(tokens):
    print("s{} ({:2d} tokens): {}".format(i, len(t), t))

assert tokens[0][:3] == ["city", "council", "approved"], "got {}".format(tokens[0][:3])
assert "the" not in tokens[0], "stopwords should be gone"
assert "barrier" in tokens[1], "real words should survive"
print("\ncheck passed")

---
## Cell 3 — Vocabulary and the count matrix  ·  *explainer section 5*

The **vocabulary** is every distinct token in the article, sorted. Sorting is not required by the maths — it just
means column 0 always refers to the same word, which makes everything debuggable.

Then the **document-term matrix**: one row per sentence, one column per vocabulary word, each cell a count.
Shape `(8, vocab_size)`.

In [ ]:
# TODO 3a: build a sorted list of every distinct token across all sentences.
#          Hint: a set comprehension over `tokens`, then sorted().
vocab = None

vocab_index = {w: j for j, w in enumerate(vocab)}   # word -> column number

counts = np.zeros((N, len(vocab)))
for i, toks in enumerate(tokens):
    for w in toks:
        # TODO 3b: add 1 to the cell for this sentence and this word.
        pass

print("vocabulary:", len(vocab), "words")
print("counts matrix shape:", counts.shape)
print("total tokens counted:", int(counts.sum()),
      "(should equal", sum(len(t) for t in tokens), ")")

assert counts.shape == (N, len(vocab))
assert counts.sum() == sum(len(t) for t in tokens), "every token must land in exactly one cell"
assert counts[1, vocab_index["barrier"]] == 1
print("check passed")

---
## Cell 4 — TF, term frequency  ·  *explainer section 5*

Divide each row by its own total, so a sentence's counts become fractions summing to 1.

**Why:** without it a long sentence has bigger numbers everywhere and wins every comparison just for being long.
This one division is what saves us from correcting for length again later.

In [ ]:
row_totals = counts.sum(axis=1, keepdims=True)   # shape (8, 1) -- keepdims lets it broadcast

# TODO 4a: divide `counts` by `row_totals` to get tf.
#          Use np.maximum(row_totals, 1) as the denominator -- it guards against a
#          divide-by-zero if some sentence ended up with no tokens at all.
tf = None

print("row sums (each should be 1.0):", tf.sum(axis=1))

assert tf.shape == counts.shape
assert np.allclose(tf.sum(axis=1), 1.0), "each row of tf must sum to 1"
print("check passed")

---
## Cell 5 — IDF, the rarity bonus  ·  *explainer section 6*

The important one. Two steps:

- **df(word)** = document frequency = *in how many sentences does this word appear at all?* Note **at all** — a
  word appearing 3 times in one sentence still has df = 1. So count non-zeros, not values.
- **idf(word)** = `log(N / df(word))`

A word in all 8 sentences gets `log(1) = 0`, so tf-idf wipes it out completely. A word in 1 sentence gets
`log(8) ≈ 2.08`.

In [ ]:
# TODO 5a: document frequency -- for each column, in how many rows is the count > 0?
#          Hint: (counts > 0) gives a boolean matrix, and summing booleans down
#          axis=0 counts the Trues.
df = None

# TODO 5b: idf = log(N / df)
idf = None

# TODO 5c: tf_idf = tf * idf
#          numpy broadcasts the length-vocab idf vector across all 8 rows for free.
tf_idf = None

print("'barrier'  df={}  idf={:.3f}   <- in 5 of 8 sentences, so low weight".format(
    df[vocab_index["barrier"]], idf[vocab_index["barrier"]]))
print("'schools'  df={}  idf={:.3f}   <- appears once, so high weight".format(
    df[vocab_index["schools"]], idf[vocab_index["schools"]]))
print("\nwords appearing in exactly one sentence:", (df == 1).sum(), "of", len(vocab))

assert df[vocab_index["barrier"]] == 5, "barrier is in 5 of the 8 sentences"
assert abs(idf[vocab_index["barrier"]] - np.log(8 / 5)) < 1e-9
assert tf_idf.shape == tf.shape
print("check passed")

### Look at what idf learned

Nothing to write. This is the table that tells you whether your idf is sane.

In [ ]:
order = np.argsort(-idf)
print("{:<14}{:>4}{:>8}".format("word", "df", "idf"))
print("-" * 26)
for j in list(order[:6]) + list(order[-6:]):
    print("{:<14}{:>4}{:>8.3f}".format(vocab[j], int(df[j]), idf[j]))
print("\nTop = distinctive words. Bottom = words in so many sentences they are nearly worthless.")

---
## Cell 6 — Methods 1 and 2: lead-3, and TF-IDF scoring  ·  *explainer section 7*

**Lead-3** is `[0, 1, 2]`. That is the whole method, and it is a genuinely hard baseline to beat on news.

**TF-IDF scoring**: a sentence's score is the sum of its row in `tf_idf`. Take the top *k*, then **sort those
indices back into document order** so the summary still reads forwards.

No length correction is needed — `tf` was already divided by sentence length in cell 4.

In [ ]:
def lead_k(k=3):
    """The do-nothing baseline: the first k sentences."""
    return list(range(k))


def top_k_indices(scores, k=3):
    """Indices of the k highest scores, returned in original document order."""
    # TODO 6a: get the indices of the k largest values in `scores`, then sort those
    #          indices ascending so the summary reads in the original order.
    #          Hint: np.argsort(-scores) ranks best-first; slice it, then sorted().
    return None


# TODO 6b: score each sentence by summing its row of tf_idf. Shape should be (8,).
tfidf_scores = None


def show(indices, title):
    print("---", title, "---")
    for i in indices:
        print("  s{}: {}".format(i, sentences[i]))
    print()


print("sentence scores:", tfidf_scores, "\n")
show(lead_k(3), "Lead-3")
show(top_k_indices(tfidf_scores, 3), "TF-IDF top 3")

assert tfidf_scores.shape == (N,)
assert top_k_indices(np.array([5.0, 1.0, 9.0, 2.0]), 2) == [0, 2], "must come back in document order"
print("check passed")

**Stop and look at the TF-IDF summary before moving on.** Compare it against the article in cell 0.

Does it tell you *who approved what*? Or has it gone for the sentences with the most unusual vocabulary? Write down
what you think went wrong. We measure it in cell 10, and the number is worse than you would guess.

---
## Cell 7 — Cosine similarity  ·  *explainer section 8*

How alike are two sentences? Treat each as an arrow and measure the **angle** between them:

$$\cos(A,B) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

1 = same direction, 0 = no words in common. We use the angle rather than the distance so a short sentence and a
long sentence about the same thing still count as similar.

**The trick that makes this one line:** scale every row to length 1 first, and then the cosine between any two rows
is *just their dot product*. So all pairs at once is `X_norm @ X_norm.T`.

In [ ]:
def cosine_similarity_matrix(X):
    """Return the (n, n) matrix of cosine similarities between the rows of X."""
    # TODO 7a: the length (L2 norm) of each row. Keep it shaped (n, 1) so it broadcasts.
    #          Hint: np.linalg.norm(X, axis=1, keepdims=True)
    norms = None

    # Guard: an all-zero row would divide by 0.
    norms = np.where(norms == 0, 1, norms)

    # TODO 7b: divide X by norms so every row has length exactly 1.
    X_norm = None

    # TODO 7c: all pairwise cosines, in one matrix multiply.
    return None


sim = cosine_similarity_matrix(tf_idf)

print("similarity matrix:")
print(sim)

assert sim.shape == (N, N)
assert np.allclose(np.diag(sim), 1.0), "a sentence is identical to itself, so the diagonal must be 1"
assert np.allclose(sim, sim.T), "similarity is symmetric"
assert (sim >= -1e-9).all(), "counts are non-negative, so cosines cannot be negative here"
print("\ncheck passed")

In [ ]:
# Nothing to write -- which two sentences are most alike?
pairs = [(sim[a, b], a, b) for a in range(N) for b in range(a + 1, N)]
pairs.sort(reverse=True)

print("most similar pairs:")
for score, a, b in pairs[:3]:
    print("  {:.3f}  s{} / s{}".format(score, a, b))

print("\ns{}: {}".format(pairs[0][1], sentences[pairs[0][1]]))
print("s{}: {}".format(pairs[0][2], sentences[pairs[0][2]]))
print("\nIf both of those land in your summary, you wasted a slot. That is the redundancy problem.")

---
## Cell 8 — TextRank  ·  *explainer section 9*

**The idea:** a sentence is important if important sentences are similar to it. Circular — but keep passing the
score around and it settles.

The money analogy: eight people each start with 1/8. Everyone splits their money among the people similar to them
and hands it over. Repeat until nothing moves.

Three things to get right:

1. **Zero the diagonal.** A sentence must not vote for itself, or it hoards its own score.
2. **Row-normalise.** Divide each row by its sum so each sentence hands out exactly 100% of what it holds. Those
   *shares* are what keeps the total conserved.
3. **Damping.** Each round, `d = 0.85` of the score flows along the edges and `(1 - d)` is shared equally among
   everyone — so no clique can trap it all.

One round, for every sentence at once:

`new = (1 - d) / N + d * (P.T @ scores)`

The `P.T` is the part people get wrong. `P[j, i]` is the share flowing **from j to i**. To collect everything
arriving **at i**, you sum down column `i` of `P` — which is row `i` of `P.T`.

### The dangling-node trap (already handled for you — but read this)

Look at **s2** in cell 7's similarity matrix: it shares no words with any other sentence, so its whole row is
zero. It has nobody to hand its money to. Divide that row by its sum and you get 0/0; leave it as zeros and s2's
score simply **vanishes from the system** every round — your total drains away from 1.0 and the final ranking is
quietly wrong.

A node with no outgoing edges is called a **dangling node**, and it is the single most common PageRank bug. The
standard fix is in the cell below: a sentence similar to nothing hands out to *everyone* equally. That is also the
honest interpretation — we have no information about where its importance should go, so we spread it evenly.

The check `scores.sum() == 1.0` at the bottom is what catches this. Trust that assert.

In [ ]:
def textrank(sim, d=0.85, max_iter=200, tol=1e-10):
    """PageRank over the sentence-similarity graph. Returns one score per sentence."""
    n = sim.shape[0]

    W = sim.copy()
    # TODO 8a: no self-votes -- set the diagonal of W to 0.
    #          Hint: np.fill_diagonal(W, 0) edits W in place.

    # Dangling nodes, handled for you: a sentence similar to nothing has an all-zero
    # row, so it has nobody to pass its score to. Let it pass to everyone equally,
    # otherwise its score leaks out of the system and the totals stop adding to 1.
    dangling = (W.sum(axis=1) == 0)
    if dangling.any():
        print("dangling sentences (similar to nothing):", np.flatnonzero(dangling).tolist())
        W[dangling] = 1.0 / n

    row_sums = W.sum(axis=1, keepdims=True)
    # TODO 8b: row-normalise W into P, so each row of P sums to 1.
    #          Divide W by row_sums.
    P = None

    scores = np.full(n, 1.0 / n)

    for iteration in range(max_iter):
        # TODO 8c: one round of the update shown above.
        new_scores = None

        if np.abs(new_scores - scores).max() < tol:
            scores = new_scores
            print("converged after", iteration + 1, "rounds")
            break
        scores = new_scores

    return scores


textrank_scores = textrank(sim)

print("\nscores:", textrank_scores)
print("sum:", textrank_scores.sum(), "(should be ~1.0 -- score is conserved)\n")

for i in np.argsort(-textrank_scores):
    print("s{} {:.4f} {}".format(i, textrank_scores[i], "#" * int(textrank_scores[i] * 120)))

assert np.isclose(textrank_scores.sum(), 1.0), "total score must stay at 1"
assert (textrank_scores > 0).all(), "damping guarantees nobody hits exactly zero"
print("\ncheck passed")

---
## Cell 9 — All three summaries, side by side

Nothing to write. Read the three outputs and form an opinion **before** you see the scores in cell 10.

In [ ]:
methods = {
    "lead-3":   lead_k(3),
    "tf-idf":   top_k_indices(tfidf_scores, 3),
    "textrank": top_k_indices(textrank_scores, 3),
}

for name, picks in methods.items():
    text = " ".join(sentences[i] for i in picks)
    print("=== {} === (sentences {}, {} words)".format(name, picks, len(text.split())))
    print(text, "\n")

print("Reference (human):")
print(REFERENCE)

---
## Cell 10 — ROUGE-1: score them  ·  *explainer section 11*

Now measure it. **ROUGE-1** counts how many single words your summary and the human reference share.

The subtlety is **clipping**. If your summary says "barrier" five times and the reference says it twice, you get
credit for **two**, not five — otherwise repeating one word would inflate your score forever. So: count how many
of each word the reference has, then spend that budget down as you walk your own tokens.

- **precision** = overlap / length of *your* summary — *was what I wrote wanted?*
- **recall** = overlap / length of the *reference* — *did I cover what was wanted?*
- **F1** = `2PR / (P + R)` — the honest single number.

In [ ]:
from collections import Counter


def rouge_1(candidate, reference):
    """Unigram overlap with clipping. Returns (precision, recall, f1)."""
    cand_tokens = tokenize(candidate)
    ref_tokens = tokenize(reference)

    ref_counts = Counter(ref_tokens)

    # TODO 10a: count the clipped overlap. Walk cand_tokens; a token scores 1 only if
    #           ref_counts still has budget left for it, and then that budget drops by 1.
    overlap = 0
    for w in cand_tokens:
        pass

    # TODO 10b: precision = overlap / len(cand_tokens)
    #           recall    = overlap / len(ref_tokens)
    #           Use 0.0 if a denominator is 0.
    precision = None
    recall = None

    # TODO 10c: f1 = 2 * precision * recall / (precision + recall), or 0.0 if both are 0.
    f1 = None

    return precision, recall, f1


# The clipping test: "barrier" five times must not earn more than the reference has.
p, r, f = rouge_1("barrier barrier barrier barrier barrier", "the barrier was approved")
assert abs(p - 1 / 5) < 1e-9, "clipping is not working -- got precision {}".format(p)

print("{:<12}{:>11}{:>9}{:>8}".format("method", "precision", "recall", "F1"))
print("-" * 40)

results = {}
for name, picks in methods.items():
    text = " ".join(sentences[i] for i in picks)
    results[name] = rouge_1(text, REFERENCE)
    p, r, f = results[name]
    print("{:<12}{:>11.3f}{:>9.3f}{:>8.3f}".format(name, p, r, f))

best = max(results, key=lambda k: results[k][2])
print("\nbest by F1:", best)
print("check passed")

---
## Cell 11 — What actually happened

Run this once your numbers are in. It states the result in words, so you can check your reading of the table
against it.

In [ ]:
lead_f = results["lead-3"][2]
tfidf_f = results["tf-idf"][2]
tr_f = results["textrank"][2]

print("lead-3   F1 = {:.3f}   <- the do-nothing baseline".format(lead_f))
print("tf-idf   F1 = {:.3f}".format(tfidf_f))
print("textrank F1 = {:.3f}".format(tr_f))
print()
print("TextRank beat the baseline." if tr_f > lead_f else "TextRank did NOT beat the baseline.")
print("TF-IDF beat the baseline." if tfidf_f > lead_f else
      "TF-IDF LOST to the baseline -- and that is the real lesson of this notebook.")

print("\nWhy TF-IDF lost: it scores each sentence alone, so it rewards rare vocabulary.")
print("The sentences it chose:")
for i in methods["tf-idf"]:
    rare = [w for w in tokens[i] if df[vocab_index[w]] == 1]
    print("  s{}: {} of {} tokens appear in no other sentence".format(i, len(rare), len(tokens[i])))

print("\nTextRank asks a different question -- is this sentence connected to the rest of the article?")
print("The sentences it chose:")
for i in methods["textrank"]:
    print("  s{}: total similarity to all the others = {:.3f}".format(i, sim[i].sum() - 1))

---
## Cell 12 — Cross-check against scikit-learn

Your tf-idf will **not** match sklearn's exactly, and that is expected rather than a bug. Three real differences:

| | yours | sklearn |
|---|---|---|
| idf formula | `log(N / df)` — the textbook one | `log((1+N)/(1+df)) + 1` — smoothed, never zero |
| row scaling | tf divided by sentence length | each row L2-normalised to length 1 |
| tokenizer | your regex, keeps 1-character tokens | `\b\w\w+\b`, silently drops every 1-character token |

So expect the **top-3 to agree on roughly two of three sentences, not all three** — and when you run it, that is
exactly what happens. Do not go hunting for a bug in your code when you see the mismatch.

That is the honest lesson of this cell: with 8 sentences, sentences ranked 3rd and 4th are separated by a hair, so a
harmless change in the idf formula is enough to swap them. Two implementations agreeing on the *clear* winners and
disagreeing on the *marginal* ones is what a small, noisy problem looks like. It is not a sign either is wrong.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(stop_words=list(STOPWORDS), lowercase=True)
X_sk = vec.fit_transform(sentences)
sim_sk = cosine_similarity(X_sk)

print("yours:  ", len(vocab), "vocabulary words")
print("sklearn:", len(vec.get_feature_names_out()), "vocabulary words")

mine = sorted(((sim[a, b], a, b) for a in range(N) for b in range(a + 1, N)), reverse=True)
theirs = sorted(((sim_sk[a, b], a, b) for a in range(N) for b in range(a + 1, N)), reverse=True)

print("\nmost similar pair -- yours: s{}/s{}   sklearn: s{}/s{}".format(
    mine[0][1], mine[0][2], theirs[0][1], theirs[0][2]))

sk_scores = textrank(sim_sk)
mine_top = top_k_indices(textrank_scores, 3)
sk_top = top_k_indices(sk_scores, 3)

print("\ntextrank top 3 -- yours: {}   on sklearn's matrix: {}".format(mine_top, sk_top))
print("agreed on {} of 3 sentences: {}".format(
    len(set(mine_top) & set(sk_top)), sorted(set(mine_top) & set(sk_top))))
print("\nTwo implementations agreeing on the clear winners and differing on the marginal one")
print("is the expected result here -- not a bug to chase.")

---
## Cell 13 — Optional: fix the redundancy problem with MMR

Only once everything above passes. **Maximal Marginal Relevance** picks sentences one at a time, each time
preferring one that is high-scoring *and* unlike what is already chosen:

`pick = argmax over remaining of [ lam * score(i) - (1 - lam) * max(sim(i, already_chosen)) ]`

`lam = 1` ignores redundancy entirely; `lam = 0.5` weighs novelty as much as importance. It is **greedy** — it
takes the best-looking option at each step and never reconsiders.

In [ ]:
def mmr(scores, sim, k=3, lam=0.7):
    """Greedily pick k sentences, balancing score against novelty."""
    chosen = []
    remaining = list(range(len(scores)))

    # Rescale scores to 0-1 so `lam` trades off two comparable quantities.
    s = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)

    while len(chosen) < k and remaining:
        best_i, best_val = None, -np.inf
        for i in remaining:
            # TODO 13a: redundancy = the largest similarity between i and anything
            #           already chosen. It is 0.0 for the very first pick.
            redundancy = None

            # TODO 13b: value = lam * s[i] - (1 - lam) * redundancy
            value = None

            if value > best_val:
                best_i, best_val = i, value

        chosen.append(best_i)
        remaining.remove(best_i)

    return sorted(chosen)


mmr_picks = mmr(textrank_scores, sim, k=3, lam=0.7)

print("textrank alone:", top_k_indices(textrank_scores, 3))
print("textrank + mmr:", mmr_picks)

p, r, f = rouge_1(" ".join(sentences[i] for i in mmr_picks), REFERENCE)
print("\nmmr ROUGE-1 F1 = {:.3f}  (textrank alone was {:.3f})".format(f, results["textrank"][2]))
print("\nMMR may score the same or slightly worse here -- eight sentences is a tiny pool.")
print("It earns its keep on long documents, where near-duplicate sentences are common.")

---
## Cell 14 — Your turn: run it on your own text

Paste in anything — a news article, a page of your own notes, a Wikipedia paragraph. The pipeline is
document-agnostic; only `ARTICLE` was ever specific to this example.

**Worth trying, and what to watch for:**

- A **conversation or interview**. Extractive methods do badly. Why? No single sentence carries the point.
- A **very short text** (3–4 sentences). TextRank gets unstable — too few nodes for the graph to have structure.
- Text with **one repeated theme**. Watch the redundancy problem appear, then try MMR on it.
- Your own **NLP revision notes**. See whether it surfaces the definitions.

In [ ]:
MY_TEXT = """
Paste your own text here. Three or more sentences, each ending in a full stop
followed by a space and a capital letter.
""".strip()


def summarize(text, k=3, method="textrank"):
    """The whole pipeline end to end. Every piece is code you already wrote."""
    sents = split_sentences(text)
    if len(sents) <= k:
        return sents

    toks = [tokenize(s) for s in sents]
    vocab_ = sorted({w for t in toks for w in t})
    vi = {w: j for j, w in enumerate(vocab_)}

    C = np.zeros((len(sents), len(vocab_)))
    for i, t in enumerate(toks):
        for w in t:
            C[i, vi[w]] += 1

    tf_ = C / np.maximum(C.sum(axis=1, keepdims=True), 1)
    df_ = (C > 0).sum(axis=0)
    X = tf_ * np.log(len(sents) / np.maximum(df_, 1))

    if method == "lead":
        picks = list(range(k))
    elif method == "tf-idf":
        picks = top_k_indices(X.sum(axis=1), k)
    else:
        picks = top_k_indices(textrank(cosine_similarity_matrix(X)), k)

    return [sents[i] for i in picks]


for line in summarize(MY_TEXT, k=3):
    print("-", line)

---
## Done. What you actually built

- A sentence splitter, a tokenizer and a document-term matrix — the front end of *every* classical NLP pipeline.
- **TF-IDF**, the most reusable idea here by a distance. It powers search ranking, spam filters and document
  clustering.
- **Cosine similarity**, which is still how vector databases and RAG retrieval compare embeddings today. Same
  formula, different vectors.
- **PageRank**, via power iteration.
- **ROUGE**, and a feel for why everyone complains about it.

## The one honest limitation

Read your best summary out loud. It is made of real sentences, so it cannot state a false fact — but it does not
*flow*, and it could never compress two facts into one clause the way a person would.

That is the wall extractive methods hit, and it is exactly where **topic 03** starts: a model that writes the
summary in its own words. Bring these ROUGE numbers with you — they are the baseline it has to beat.